# 10 — Modal SuperAnimal Inference

Runs SuperAnimal-quadruped on `side.mp4` (session 1055240613) on a cloud A100 via Modal.

**One-time setup (run once per machine):**
```bash
pip install modal
modal token new      # opens browser → log in → token saved automatically
```

**What this notebook does:**
1. Builds a cloud container with DLC + models baked in (~5 min first time, cached after)
2. Downloads `side.mp4` from Allen S3 directly into the cloud machine (no local upload)
3. Runs inference on an A100 (~1.5-2 hours, ~\$2 from your \$30 credits)
4. Saves the H5 to a Modal Volume and downloads it to `outputs/pose/`

**Cost:** ~\$1.10/hr on A100 → ~\$2-2.50 per run

In [ ]:
# ── Install Modal if needed ───────────────────────────────────────────────────
import subprocess, sys
subprocess.call([sys.executable, "-m", "pip", "install", "-q", "modal"])

import modal
print(f"Modal {modal.__version__} ready")

# If you haven't authenticated yet, run in your terminal:
#   modal token new
# Then re-run this cell.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

SESSION_ID   = 1055240613
OUTPUT_LOCAL = ROOT / "outputs" / "pose" / f"session_{SESSION_ID}_superanimal.h5"
OUTPUT_LOCAL.parent.mkdir(parents=True, exist_ok=True)

print(f"Output will be saved to: {OUTPUT_LOCAL}")

In [ ]:
# ── Modal image — DLC + models baked in (cached after first build) ────────────
import modal

def _prebake_models():
    """Run during image build to cache model weights (~300 MB)."""
    from deeplabcut.pose_estimation_pytorch.modelzoo.utils import get_super_animal_snapshot_path
    for m in ["hrnet_w32", "fasterrcnn_resnet50_fpn_v2"]:
        print(f"Downloading {m} ...")
        p = get_super_animal_snapshot_path("superanimal_quadruped", m)
        print(f"  OK: {p}")

dlc_image = (
    modal.Image.debian_slim(python_version="3.11")
    .apt_install(
        "ffmpeg",
        "libgl1",
        "libglib2.0-0",
        "libsm6",
        "libxext6",
        "libxrender1",
    )
    .pip_install("numba>=0.64", extra_options="--prefer-binary")
    .pip_install("deeplabcut>=3.0.0rc1", extra_options="--no-deps --prefer-binary")
    .pip_install("filterpy", "ruamel.yaml", "munkres", "dlclibrary", "boto3")
    .run_function(_prebake_models)
)

print("Image defined — will build on first run (5-10 min, cached after)")

In [ ]:
# ── Modal app + volume ────────────────────────────────────────────────────────
import modal

app    = modal.App("vbn-superanimal", image=dlc_image)
volume = modal.Volume.from_name("vbn-superanimal-output", create_if_missing=True)

print("App and volume defined")

In [ ]:
# ── Inference function ────────────────────────────────────────────────────────
import modal

@app.function(
    gpu="L40S",
    cpu=4.0,
    memory=16384,
    timeout=10800,
    volumes={"/output": volume},
)
def run_superanimal(session_id: int = 1055240613) -> str:
    import os, re, json, sys, shutil, tempfile, subprocess
    import torch
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config
    from pathlib import Path

    VIDEO  = Path("/tmp/side.mp4")
    OUTPUT = Path(f"/output/session_{session_id}_superanimal.h5")

    # ── Download video from Allen S3 ──────────────────────────────────────────
    s3_key = f"visual-behavior-neuropixels/raw-data/{session_id}/behavior_videos/side.mp4"
    s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

    meta           = s3.head_object(Bucket="allen-brain-observatory", Key=s3_key)
    expected_bytes = meta["ContentLength"]
    print(f"Downloading side.mp4 ({expected_bytes/1024**3:.2f} GB) ...", flush=True)

    obj  = s3.get_object(Bucket="allen-brain-observatory", Key=s3_key)
    body = obj["Body"]
    total = 0
    with open(VIDEO, "wb") as f:
        while chunk := body.read(16 * 1024 * 1024):
            f.write(chunk)
            total += len(chunk)
            print(f"\r  {total/1024**3:.2f}/{expected_bytes/1024**3:.2f} GB", end="", flush=True)
    print()

    actual = VIDEO.stat().st_size
    if actual != expected_bytes:
        raise RuntimeError(f"Download size mismatch: {actual} vs {expected_bytes}")
    print(f"Download complete: {actual/1024**3:.2f} GB", flush=True)

    # ── GPU info + adaptive batch size ────────────────────────────────────────
    import deeplabcut as dlc
    free_gb  = torch.cuda.mem_get_info(0)[0] / 1024**3
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    det_b    = max(4, int((free_gb - 2.0) / 0.4))
    print(f"GPU: {torch.cuda.get_device_name(0)}  "
          f"free={free_gb:.1f}/{total_gb:.1f} GB  det_b={det_b}", flush=True)

    # ── Patch DLC detector batch size ─────────────────────────────────────────
    pattern = re.compile(r'get\("detector_batch_size",\s*\d+\)')
    replacement = f'get("detector_batch_size", {det_b})'
    for fname in [
        "pose_estimation_pytorch/apis/videos.py",
        "pose_estimation_pytorch/apis/tracking_dataset.py",
    ]:
        fpath = Path(dlc.__path__[0]) / fname
        if fpath.exists():
            content = fpath.read_text()
            if pattern.search(content):
                fpath.write_text(pattern.sub(replacement, content))
                for pyc in (fpath.parent / "__pycache__").glob(f"{fpath.stem}.*.pyc"):
                    pyc.unlink(missing_ok=True)
                print(f"Patched {fname}", flush=True)

    # ── Run inference in subprocess (picks up patched files fresh) ────────────
    args = {
        "videos":              [str(VIDEO)],
        "dest_folder":         "/tmp",
        "batch_size":          16,
        "detector_batch_size": det_b,
        "device":              "cuda",
    }
    runner_src = f"""\
import os, sys, json
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import deeplabcut as dlc
args = {json.dumps(args)}
print(f"Running det_b={{args['detector_batch_size']}} pose_b={{args['batch_size']}}", flush=True)
dlc.video_inference_superanimal(
    videos=args["videos"],
    superanimal_name="superanimal_quadruped",
    model_name="hrnet_w32",
    detector_name="fasterrcnn_resnet50_fpn_v2",
    scale_list=[200],
    videotype="mp4",
    dest_folder=args["dest_folder"],
    create_labeled_video=False,
    plot_trajectories=False,
    batch_size=args["batch_size"],
    detector_batch_size=args["detector_batch_size"],
    device=args["device"],
)
"""
    runner = tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False)
    runner.write(runner_src)
    runner.close()
    try:
        ret = subprocess.call([sys.executable, runner.name])
    finally:
        os.unlink(runner.name)

    if ret != 0:
        raise RuntimeError(f"Inference subprocess exited {ret} — check logs above")

    # ── Move H5 to volume ─────────────────────────────────────────────────────
    h5s = sorted(Path("/tmp").glob("side*superanimal*.h5"))
    if not h5s:
        h5s = sorted(Path("/tmp").glob("side*.h5"))
    if not h5s:
        raise RuntimeError("No .h5 output found in /tmp — inference may have failed silently")

    shutil.move(str(h5s[0]), str(OUTPUT))
    volume.commit()

    size_mb = OUTPUT.stat().st_size / 1024**2
    if size_mb < 1:
        raise RuntimeError(f"Output H5 only {size_mb:.1f} MB — suspiciously small")

    print(f"\nSaved to volume: {OUTPUT} ({size_mb:.0f} MB)", flush=True)
    return f"session_{session_id}_superanimal.h5"

print("Inference function defined — run the next cell to launch")

In [ ]:
# ── Launch — this cell blocks until the job completes (~1.5-2 hours) ─────────
# First run also builds the image (~5-10 min). Subsequent runs skip that.
#
# Monitor live logs at: https://modal.com/apps

import modal

print("Launching on Modal A100 ...")
print("Live logs: https://modal.com/apps\n")

with modal.enable_output():
    with app.run():
        output_filename = run_superanimal.remote(SESSION_ID)

print(f"\nJob complete: {output_filename}")

In [ ]:
# ── Download H5 from Modal volume to local outputs/pose/ ──────────────────────
import modal

volume = modal.Volume.from_name("vbn-superanimal-output", create_if_missing=False)

print(f"Downloading {output_filename} from volume ...")
with open(OUTPUT_LOCAL, "wb") as f:
    for chunk in volume.read_file(output_filename):
        f.write(chunk)

size_mb = OUTPUT_LOCAL.stat().st_size / 1024**2
print(f"Saved: {OUTPUT_LOCAL} ({size_mb:.0f} MB)")
print("\nReady for NB07.")

In [ ]:
# ── Quick sanity check on the H5 ─────────────────────────────────────────────
import pandas as pd
import numpy as np

df = pd.read_hdf(OUTPUT_LOCAL)
print(f"Shape: {df.shape}  — expected (~575526, ~81)")
print(f"Columns (first 6): {df.columns.tolist()[:6]}")

# Detection rate per keypoint (likelihood > 0.6)
like_cols = [c for c in df.columns if "likelihood" in str(c).lower()]
if like_cols:
    detection_rate = (df[like_cols] > 0.6).mean()
    print(f"\nDetection rate (likelihood > 0.6):")
    print(detection_rate.to_string())
    print(f"\nMean across all keypoints: {detection_rate.mean():.1%}")
else:
    print("Note: likelihood columns have multi-level index — check df.columns")